# CrisisMap ML Fine-Tuning — Stage A: xBD/xView2

**Objective:** Fine-tune the Microsoft SiamUnet checkpoint on xBD challenge data to improve building damage classification.

**Platform:** Kaggle (GPU P100/T4) — mount xBD as private Kaggle Dataset

**Strategy:**
- Phase 1: Train segmentation arm (freeze damage classifier)
- Phase 2: Train damage classifier (freeze segmentation)

**Verified Against:**
- `end_to_end_Siam_UNet.py`: forward(x1,x2) → (seg_pre, seg_post, damage)
- `ml_inference.py`: RGB [1,2,3], normalize /255 → [0,1], 256×256
- `model_best.pth.tar`: key='state_dict', 176 params, epoch=279, best_f1=0.6227

## 0. Setup & Configuration

In [ ]:
# ============================================================================
# CELL 0: Environment Setup & Reproducibility
# ============================================================================
import os
import sys
import json
import time
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import yaml

warnings.filterwarnings('ignore')

# --- Reproducibility: set ALL random seeds ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- Device ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')
print(f'NumPy: {np.__version__}')

In [ ]:
# ============================================================================
# CELL 1: Detect Platform & Set Paths
# ============================================================================

# Auto-detect: Kaggle vs Local
IS_KAGGLE = os.path.exists('/kaggle/input')
IS_LOCAL = not IS_KAGGLE

if IS_KAGGLE:
    # ---- Auto-detect xBD paths ----
    # Kaggle extracts tar.gz with varying subfolder depth.
    # Scan all dirs under /kaggle/input to find ones with images/ + labels/
    def find_xbd_root(base, must_contain_both=('images', 'labels'), prefer_keyword=None):
        best = None
        for root, dirs, files in os.walk(base):
            dirs_set = set(d.lower() for d in dirs)
            if 'images' in dirs_set and 'labels' in dirs_set:
                p = root
                if prefer_keyword and prefer_keyword.lower() not in p.lower():
                    # Store as fallback but keep looking
                    if best is None:
                        best = Path(p)
                    continue
                return Path(p)
        return best  # return best fallback
    
    XBD_TRAIN_ROOT = find_xbd_root('/kaggle/input', prefer_keyword='train')
    XBD_TIER3_ROOT = find_xbd_root('/kaggle/input', prefer_keyword='tier3')
    
    # ---- Auto-detect checkpoint ----
    def find_checkpoint(base):
        for root, dirs, files in os.walk(base):
            for fname in files:
                if fname.endswith('.pth.tar') or fname.endswith('.pth'):
                    return Path(root) / fname
        return None
    
    CHECKPOINT_PATH = find_checkpoint('/kaggle/input')
    OUTPUT_DIR = Path('/kaggle/working/output')
else:
    # --- Local paths (Windows) ---
    XBD_TRAIN_ROOT = Path('E:/CrisisMapData/xbd/extracted/train/train')
    XBD_TIER3_ROOT = Path('E:/CrisisMapData/xbd/extracted/tier3/tier3')
    CHECKPOINT_PATH = Path(r'c:/Users/HP/Laravel/CrisisMap/data/raw/models/microsoft_building_damage_assessment_cnn_siamese/model_best.pth.tar')
    OUTPUT_DIR = Path('./output')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Platform: {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Train root: {XBD_TRAIN_ROOT} (exists: {XBD_TRAIN_ROOT.exists() if XBD_TRAIN_ROOT else False})')
print(f'Tier3 root: {XBD_TIER3_ROOT} (exists: {XBD_TIER3_ROOT.exists() if XBD_TIER3_ROOT else False})')
print(f'Checkpoint: {CHECKPOINT_PATH} (exists: {CHECKPOINT_PATH.exists() if CHECKPOINT_PATH else False})')
print(f'Output dir: {OUTPUT_DIR}')

# Safety check
if XBD_TRAIN_ROOT is None:
    raise RuntimeError('Could not find xBD train data! Make sure xbd-xview2-challenge dataset is attached.')
if XBD_TIER3_ROOT is None:
    print('WARNING: Tier3 not found. Will use train data only.')
    XBD_TIER3_ROOT = XBD_TRAIN_ROOT  # fallback
if CHECKPOINT_PATH is None:
    raise RuntimeError('Could not find checkpoint (.pth.tar)! Make sure crisismap-checkpoint dataset is attached.')


In [ ]:
# ============================================================================
# CELL 2: Load Configuration
# ============================================================================

# Inline config (so notebook is self-contained on Kaggle)
CONFIG = {
    'seed': 42,
    'model': {
        'in_channels': 3,
        'out_channels_s': 2,
        'out_channels_c': 5,
        'init_features': 16,
    },
    'preprocessing': {
        'tile_size': 1024,
        'patch_size': 256,
        'patches_per_tile': 16,
        'overlap_patches': 0,  # start without overlap for speed
        'normalize_value': 255.0,
    },
    'split': {
        'train_ratio': 0.80,
        'val_ratio': 0.10,
        'test_ratio': 0.10,
    },
    'stage_a': {
        'phase1': {
            'epochs': 15,
            'lr': 1e-4,
            'weight_decay': 1e-5,
            'batch_size': 16,
            'num_workers': 2,
            'weights_loss': [0.5, 0.5, 0.0],
            'class_weights_seg': [0.3, 0.7],
        },
        'phase2': {
            'epochs': 20,
            'lr': 5e-5,
            'weight_decay': 1e-5,
            'batch_size': 16,
            'num_workers': 2,
            'weights_loss': [0.0, 0.0, 1.0],
            'class_weights_dmg': [0.05, 0.10, 0.60, 0.80, 1.00],
        },
    },
    'compute': {
        'subsample_tier3': 0.5,  # use 50% of tier3 initially
        'checkpoint_save_interval': 500,  # steps
        'log_interval': 50,  # steps
    },
}

# Damage class names
DAMAGE_CLASSES = ['background', 'no-damage', 'minor-damage', 'major-damage', 'destroyed']
SEG_CLASSES = ['background', 'building']

# Freeze layer prefixes
DAMAGE_LAYER_PREFIXES = [
    'upconv4_c', 'conv4_c', 'upconv3_c', 'conv3_c',
    'upconv2_c', 'conv2_c', 'upconv1_c', 'conv1_c', 'conv_c'
]

print('Config loaded.')
print(f'  Phase 1: {CONFIG["stage_a"]["phase1"]["epochs"]} epochs, lr={CONFIG["stage_a"]["phase1"]["lr"]}')
print(f'  Phase 2: {CONFIG["stage_a"]["phase2"]["epochs"]} epochs, lr={CONFIG["stage_a"]["phase2"]["lr"]}')
print(f'  Subsample tier3: {CONFIG["compute"]["subsample_tier3"]}')

## 1. Model Architecture (SiamUnet)

In [ ]:
# ============================================================================
# CELL 3: SiamUnet Model Definition
# ============================================================================
# Copied verbatim from end_to_end_Siam_UNet.py (Microsoft/NLRC 510)
# DO NOT MODIFY — must match backend exactly for checkpoint compatibility

from collections import OrderedDict

class SiamUnet(nn.Module):

    def __init__(self, in_channels=3, out_channels_s=2, out_channels_c=5, init_features=16):
        super(SiamUnet, self).__init__()

        features = init_features
        
        # UNet layers
        self.encoder1 = SiamUnet._block(in_channels, features, name="enc1")
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = SiamUnet._block(features, features * 2, name="enc2")
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder3 = SiamUnet._block(features * 2, features * 4, name="enc3")
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder4 = SiamUnet._block(features * 4, features * 8, name="enc4")
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = SiamUnet._block(features * 8, features * 16, name="bottleneck")

        self.upconv4 = nn.ConvTranspose2d(features * 16, features * 8, kernel_size=2, stride=2)
        self.decoder4 = SiamUnet._block((features * 8) * 2, features * 8, name="dec4")
        self.upconv3 = nn.ConvTranspose2d(features * 8, features * 4, kernel_size=2, stride=2)
        self.decoder3 = SiamUnet._block((features * 4) * 2, features * 4, name="dec3")
        self.upconv2 = nn.ConvTranspose2d(features * 4, features * 2, kernel_size=2, stride=2)
        self.decoder2 = SiamUnet._block((features * 2) * 2, features * 2, name="dec2")
        self.upconv1 = nn.ConvTranspose2d(features * 2, features, kernel_size=2, stride=2)
        self.decoder1 = SiamUnet._block(features * 2, features, name="dec1")
        
        self.conv_s = nn.Conv2d(in_channels=features, out_channels=out_channels_s, kernel_size=1)
        
        # Siamese classifier layers
        self.upconv4_c = nn.ConvTranspose2d(features * 16, features * 8, kernel_size=2, stride=2)
        self.conv4_c = SiamUnet._block(features * 16, features * 16, name="conv4")

        self.upconv3_c = nn.ConvTranspose2d(features * 16, features * 4, kernel_size=2, stride=2)
        self.conv3_c = SiamUnet._block(features * 8, features * 8, name="conv3")

        self.upconv2_c = nn.ConvTranspose2d(features * 8, features * 2, kernel_size=2, stride=2)
        self.conv2_c = SiamUnet._block(features * 4, features * 4, name="conv2")

        self.upconv1_c = nn.ConvTranspose2d(features * 4, features, kernel_size=2, stride=2)
        self.conv1_c = SiamUnet._block(features * 2, features * 2, name="conv1")

        self.conv_c = nn.Conv2d(in_channels=features * 2, out_channels=out_channels_c, kernel_size=1)


    def forward(self, x1, x2):
        
        # UNet on x1
        enc1_1 = self.encoder1(x1)
        enc2_1 = self.encoder2(self.pool1(enc1_1))
        enc3_1 = self.encoder3(self.pool2(enc2_1))
        enc4_1 = self.encoder4(self.pool3(enc3_1))

        bottleneck_1 = self.bottleneck(self.pool4(enc4_1))

        dec4_1 = self.upconv4(bottleneck_1)
        dec4_1 = torch.cat((dec4_1, enc4_1), dim=1)
        dec4_1 = self.decoder4(dec4_1)
        dec3_1 = self.upconv3(dec4_1)
        dec3_1 = torch.cat((dec3_1, enc3_1), dim=1)
        dec3_1 = self.decoder3(dec3_1)
        dec2_1 = self.upconv2(dec3_1)
        dec2_1 = torch.cat((dec2_1, enc2_1), dim=1)
        dec2_1 = self.decoder2(dec2_1)
        dec1_1 = self.upconv1(dec2_1)
        dec1_1 = torch.cat((dec1_1, enc1_1), dim=1)
        dec1_1 = self.decoder1(dec1_1)
        
        # UNet on x2
        enc1_2 = self.encoder1(x2)
        enc2_2 = self.encoder2(self.pool1(enc1_2))
        enc3_2 = self.encoder3(self.pool2(enc2_2))
        enc4_2 = self.encoder4(self.pool3(enc3_2))

        bottleneck_2 = self.bottleneck(self.pool4(enc4_2))

        dec4_2 = self.upconv4(bottleneck_2)
        dec4_2 = torch.cat((dec4_2, enc4_2), dim=1)
        dec4_2 = self.decoder4(dec4_2)
        dec3_2 = self.upconv3(dec4_2)
        dec3_2 = torch.cat((dec3_2, enc3_2), dim=1)
        dec3_2 = self.decoder3(dec3_2)
        dec2_2 = self.upconv2(dec3_2)
        dec2_2 = torch.cat((dec2_2, enc2_2), dim=1)
        dec2_2 = self.decoder2(dec2_2)
        dec1_2 = self.upconv1(dec2_2)
        dec1_2 = torch.cat((dec1_2, enc1_2), dim=1)
        dec1_2 = self.decoder1(dec1_2)
        
        # Siamese
        dec1_c = bottleneck_2 - bottleneck_1
        
        dec1_c = self.upconv4_c(dec1_c)
        diff_2 = enc4_2 - enc4_1
        dec2_c = torch.cat((diff_2, dec1_c), dim=1)
        dec2_c = self.conv4_c(dec2_c)
        
        dec2_c = self.upconv3_c(dec2_c)
        diff_3 = enc3_2 - enc3_1
        dec3_c = torch.cat((diff_3, dec2_c), dim=1)
        dec3_c = self.conv3_c(dec3_c)

        dec3_c = self.upconv2_c(dec3_c)
        diff_4 = enc2_2 - enc2_1
        dec4_c = torch.cat((diff_4, dec3_c), dim=1)
        dec4_c = self.conv2_c(dec4_c)

        dec4_c = self.upconv1_c(dec4_c)
        diff_5 = enc1_2 - enc1_1
        dec5_c = torch.cat((diff_5, dec4_c), dim=1)
        dec5_c = self.conv1_c(dec5_c)
        
        return self.conv_s(dec1_1), self.conv_s(dec1_2), self.conv_c(dec5_c)

    @staticmethod
    def _block(in_channels, features, name):
        return nn.Sequential(
            OrderedDict(
                [
                    (
                        name + "conv1",
                        nn.Conv2d(
                            in_channels=in_channels,
                            out_channels=features,
                            kernel_size=3,
                            padding=1,
                            bias=False,
                        ),
                    ),
                    (name + "norm1", nn.BatchNorm2d(num_features=features)),
                    (name + "relu1", nn.ReLU(inplace=True)),
                    (
                        name + "conv2",
                        nn.Conv2d(
                            in_channels=features,
                            out_channels=features,
                            kernel_size=3,
                            padding=1,
                            bias=False,
                        ),
                    ),
                    (name + "norm2", nn.BatchNorm2d(num_features=features)),
                    (name + "relu2", nn.ReLU(inplace=True)),
                ]
            )
        )

# Verify model
model = SiamUnet()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'SiamUnet loaded: {total_params:,} total params, {trainable_params:,} trainable')

# Quick forward pass test
x1 = torch.randn(1, 3, 256, 256)
x2 = torch.randn(1, 3, 256, 256)
model.eval()
with torch.no_grad():
    s1, s2, d = model(x1, x2)
print(f'Output shapes: seg_pre={s1.shape}, seg_post={s2.shape}, damage={d.shape}')
assert s1.shape == (1, 2, 256, 256), f'Expected (1,2,256,256), got {s1.shape}'
assert d.shape == (1, 5, 256, 256), f'Expected (1,5,256,256), got {d.shape}'
print('Architecture verified!')

## 2. Dataset Loading & Preprocessing

In [ ]:
# ============================================================================
# CELL 4: Dataset Utilities (self-contained for Kaggle)
# ============================================================================
# These are inlined from ml-training/utils/dataset.py to avoid import issues

import re
from PIL import Image, ImageDraw

DAMAGE_LABEL_MAP = {
    'un-classified': 0,
    'no-damage': 1,
    'minor-damage': 2,
    'major-damage': 3,
    'destroyed': 4,
}

TILE_PATTERN = re.compile(r'^(?P<disaster>.+?)_(?P<tile_id>\d+)_(?P<phase>pre|post)_disaster$')


def parse_tile_name(filename):
    stem = Path(filename).stem
    match = TILE_PATTERN.match(stem)
    if not match:
        return None
    return {
        'disaster': match.group('disaster'),
        'tile_id': match.group('tile_id'),
        'phase': match.group('phase'),
        'stem': stem,
    }


def discover_tile_pairs(data_root, images_dir='images', labels_dir='labels'):
    root = Path(data_root)
    img_dir = root / images_dir
    lbl_dir = root / labels_dir
    
    tiles = defaultdict(dict)
    for f in sorted(img_dir.iterdir()):
        if f.suffix.lower() != '.png':
            continue
        parsed = parse_tile_name(f.name)
        if parsed is None:
            continue
        key = (parsed['disaster'], parsed['tile_id'])
        tiles[key][f'img_{parsed["phase"]}'] = f
    
    for f in sorted(lbl_dir.iterdir()):
        if f.suffix.lower() != '.json':
            continue
        parsed = parse_tile_name(f.name)
        if parsed is None:
            continue
        key = (parsed['disaster'], parsed['tile_id'])
        tiles[key][f'lbl_{parsed["phase"]}'] = f
    
    pairs = []
    for (disaster, tile_id), files in sorted(tiles.items()):
        if 'img_pre' not in files or 'img_post' not in files:
            continue
        if 'lbl_post' not in files:
            continue
        pairs.append({
            'disaster': disaster,
            'tile_id': tile_id,
            'img_pre': files['img_pre'],
            'img_post': files['img_post'],
            'lbl_pre': files.get('lbl_pre'),
            'lbl_post': files['lbl_post'],
        })
    return pairs


def parse_wkt_polygon(wkt):
    try:
        inner = wkt.split('((')[1].split('))')[0]
        coords = []
        for pair in inner.split(','):
            parts = pair.strip().split()
            if len(parts) >= 2:
                coords.append((float(parts[0]), float(parts[1])))
        return coords
    except (IndexError, ValueError):
        return None


def parse_label_json(label_path):
    with open(label_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    xy_features = data.get('features', {}).get('xy', [])
    buildings = []
    for feat in xy_features:
        props = feat.get('properties', {})
        if props.get('feature_type') != 'building':
            continue
        coords = parse_wkt_polygon(feat.get('wkt', ''))
        if coords is None or len(coords) < 3:
            continue
        buildings.append({
            'subtype': props.get('subtype', 'un-classified'),
            'xy_coords': coords,
        })
    return buildings


def rasterize_buildings(buildings, height=1024, width=1024, mode='damage'):
    mask = Image.new('L', (width, height), 0)
    draw = ImageDraw.Draw(mask)
    for bld in buildings:
        coords = bld['xy_coords']
        if len(coords) < 3:
            continue
        flat_coords = [(int(round(x)), int(round(y))) for x, y in coords]
        if mode == 'segmentation':
            fill_value = 1
        else:
            fill_value = DAMAGE_LABEL_MAP.get(bld.get('subtype', 'un-classified'), 0)
        draw.polygon(flat_coords, fill=fill_value)
    return np.array(mask, dtype=np.int64)


def stratified_split(tile_pairs, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    rng = np.random.RandomState(seed)
    by_disaster = defaultdict(list)
    for pair in tile_pairs:
        by_disaster[pair['disaster']].append(pair)
    
    splits = {'train': [], 'val': [], 'test': []}
    for disaster, pairs in sorted(by_disaster.items()):
        indices = np.arange(len(pairs))
        rng.shuffle(indices)
        n = len(pairs)
        n_val = max(1, int(round(n * val_ratio)))
        n_test = max(1, int(round(n * test_ratio)))
        n_train = n - n_val - n_test
        if n_train < 1:
            n_train, n_val, n_test = n, 0, 0
        for i, idx in enumerate(indices):
            if i < n_train:
                splits['train'].append(pairs[idx])
            elif i < n_train + n_val:
                splits['val'].append(pairs[idx])
            else:
                splits['test'].append(pairs[idx])
    return splits


print('Dataset utilities loaded.')

In [ ]:
# ============================================================================
# CELL 5: XBD Patch Dataset Class
# ============================================================================

class XBDPatchDataset(torch.utils.data.Dataset):
    """PyTorch Dataset for xBD tile patches.
    
    Preprocessing matches backend ml_inference.py:
      - RGB band order
      - Normalize: img / 255.0 -> [0, 1]
      - Shape: (3, 256, 256)
    """
    
    def __init__(self, tile_pairs, patch_size=256, augment=False, use_target_png=False, seed=42):
        self.tile_pairs = tile_pairs
        self.patch_size = patch_size
        self.augment = augment
        self.use_target_png = use_target_png
        self.grid_size = 1024 // patch_size  # 4
        self.patches_per_tile = self.grid_size ** 2  # 16
        
        # Build patch index
        self._patches = []
        for tile_idx in range(len(tile_pairs)):
            for row in range(self.grid_size):
                for col in range(self.grid_size):
                    y = row * patch_size
                    x = col * patch_size
                    self._patches.append((tile_idx, y, x))
    
    def __len__(self):
        return len(self._patches)
    
    def __getitem__(self, idx):
        tile_idx, y, x = self._patches[idx]
        pair = self.tile_pairs[tile_idx]
        ps = self.patch_size
        
        # Load & normalize images (matches backend: RGB, /255.0, [0,1])
        pre_img = np.array(Image.open(pair['img_pre']).convert('RGB'), dtype=np.float32) / 255.0
        post_img = np.array(Image.open(pair['img_post']).convert('RGB'), dtype=np.float32) / 255.0
        
        # Crop patch
        pre_patch = pre_img[y:y+ps, x:x+ps].transpose(2, 0, 1)  # HWC -> CHW
        post_patch = post_img[y:y+ps, x:x+ps].transpose(2, 0, 1)
        
        # Load labels
        seg_pre, seg_post, damage = self._load_labels(pair, y, x, ps)
        
        # Augmentation
        if self.augment:
            pre_patch, post_patch, seg_pre, seg_post, damage = \
                self._augment(pre_patch, post_patch, seg_pre, seg_post, damage)
        
        return {
            'pre': torch.from_numpy(pre_patch.copy()).float(),
            'post': torch.from_numpy(post_patch.copy()).float(),
            'seg_pre': torch.from_numpy(seg_pre.copy()).long(),
            'seg_post': torch.from_numpy(seg_post.copy()).long(),
            'damage': torch.from_numpy(damage.copy()).long(),
        }
    
    def _load_labels(self, pair, y, x, ps):
        # Try target PNG first (train only)
        if self.use_target_png:
            result = self._try_target_png(pair, y, x, ps)
            if result is not None:
                return result
        
        # Rasterize from JSON
        if pair.get('lbl_pre') is not None:
            bld_pre = parse_label_json(pair['lbl_pre'])
            seg_pre = rasterize_buildings(bld_pre, mode='segmentation')[y:y+ps, x:x+ps]
        else:
            seg_pre = np.zeros((ps, ps), dtype=np.int64)
        
        bld_post = parse_label_json(pair['lbl_post'])
        seg_post = rasterize_buildings(bld_post, mode='segmentation')[y:y+ps, x:x+ps]
        damage = rasterize_buildings(bld_post, mode='damage')[y:y+ps, x:x+ps]
        
        return seg_pre, seg_post, damage
    
    def _try_target_png(self, pair, y, x, ps):
        data_root = pair['img_pre'].parent.parent
        pre_target = data_root / 'targets' / f"{pair['img_pre'].stem}_target.png"
        post_target = data_root / 'targets' / f"{pair['img_post'].stem}_target.png"
        
        if not pre_target.exists() or not post_target.exists():
            return None
        
        pre_arr = np.array(Image.open(pre_target), dtype=np.int64)
        post_arr = np.array(Image.open(post_target), dtype=np.int64)
        
        seg_pre = (pre_arr > 0).astype(np.int64)[y:y+ps, x:x+ps]
        seg_post = (post_arr > 0).astype(np.int64)[y:y+ps, x:x+ps]
        damage = post_arr[y:y+ps, x:x+ps]
        
        return seg_pre, seg_post, damage
    
    def _augment(self, pre, post, seg_pre, seg_post, damage):
        # Random horizontal flip
        if random.random() > 0.5:
            pre = np.flip(pre, axis=2).copy()
            post = np.flip(post, axis=2).copy()
            seg_pre = np.flip(seg_pre, axis=1).copy()
            seg_post = np.flip(seg_post, axis=1).copy()
            damage = np.flip(damage, axis=1).copy()
        
        # Random vertical flip  
        if random.random() > 0.5:
            pre = np.flip(pre, axis=1).copy()
            post = np.flip(post, axis=1).copy()
            seg_pre = np.flip(seg_pre, axis=0).copy()
            seg_post = np.flip(seg_post, axis=0).copy()
            damage = np.flip(damage, axis=0).copy()
        
        # Random 90-degree rotation
        k = random.randint(0, 3)
        if k > 0:
            pre = np.rot90(pre, k, axes=(1, 2)).copy()
            post = np.rot90(post, k, axes=(1, 2)).copy()
            seg_pre = np.rot90(seg_pre, k, axes=(0, 1)).copy()
            seg_post = np.rot90(seg_post, k, axes=(0, 1)).copy()
            damage = np.rot90(damage, k, axes=(0, 1)).copy()
        
        return pre, post, seg_pre, seg_post, damage

print('XBDPatchDataset defined.')

In [ ]:
# ============================================================================
# CELL 6: Discover Data & Split
# ============================================================================

print('Discovering tile pairs...')
t0 = time.time()

train_pairs = discover_tile_pairs(XBD_TRAIN_ROOT)
print(f'  Train: {len(train_pairs)} tile pairs')

tier3_pairs = discover_tile_pairs(XBD_TIER3_ROOT)
print(f'  Tier3: {len(tier3_pairs)} tile pairs')

# Subsample tier3 for initial run
subsample = CONFIG['compute']['subsample_tier3']
if subsample < 1.0:
    rng = np.random.RandomState(SEED)
    n_keep = int(len(tier3_pairs) * subsample)
    indices = rng.choice(len(tier3_pairs), n_keep, replace=False)
    tier3_pairs = [tier3_pairs[i] for i in sorted(indices)]
    print(f'  Tier3 subsampled to {len(tier3_pairs)} pairs ({subsample*100:.0f}%)')

all_pairs = train_pairs + tier3_pairs
print(f'  Total: {len(all_pairs)} tile pairs')

# Stratified split by disaster type
splits = stratified_split(
    all_pairs,
    train_ratio=CONFIG['split']['train_ratio'],
    val_ratio=CONFIG['split']['val_ratio'],
    test_ratio=CONFIG['split']['test_ratio'],
    seed=SEED,
)

print(f'\nSplit results (tile-level, stratified by disaster):')
for split_name, pairs in splits.items():
    n_tiles = len(pairs)
    n_patches = n_tiles * 16
    disasters = set(p['disaster'] for p in pairs)
    print(f'  {split_name:5s}: {n_tiles:5d} tiles -> {n_patches:6d} patches ({len(disasters)} disaster types)')

print(f'\nDiscovery took {time.time() - t0:.1f}s')

In [ ]:
# ============================================================================
# CELL 7: Create DataLoaders & Sanity Check
# ============================================================================

# Determine if train data has targets/ for faster loading
has_targets = (XBD_TRAIN_ROOT / 'targets').exists()
print(f'Train has targets/ folder: {has_targets}')
print(f'Tier3 has targets/ folder: {(XBD_TIER3_ROOT / "targets").exists()}')

# Create datasets
train_dataset = XBDPatchDataset(
    splits['train'],
    patch_size=256,
    augment=True,
    use_target_png=False,  # Use JSON rasterization for consistency (tier3 has no targets/)
)
val_dataset = XBDPatchDataset(
    splits['val'],
    patch_size=256,
    augment=False,
    use_target_png=False,
)
test_dataset = XBDPatchDataset(
    splits['test'],
    patch_size=256,
    augment=False,
    use_target_png=False,
)

print(f'\nDataset sizes:')
print(f'  Train: {len(train_dataset):,} patches')
print(f'  Val:   {len(val_dataset):,} patches')
print(f'  Test:  {len(test_dataset):,} patches')

# Sanity check: load one batch
print('\nLoading sanity check batch...')
sample = train_dataset[0]
print(f'  pre:      {sample["pre"].shape}, dtype={sample["pre"].dtype}, range=[{sample["pre"].min():.3f}, {sample["pre"].max():.3f}]')
print(f'  post:     {sample["post"].shape}, dtype={sample["post"].dtype}')
print(f'  seg_pre:  {sample["seg_pre"].shape}, unique={torch.unique(sample["seg_pre"]).tolist()}')
print(f'  seg_post: {sample["seg_post"].shape}, unique={torch.unique(sample["seg_post"]).tolist()}')
print(f'  damage:   {sample["damage"].shape}, unique={torch.unique(sample["damage"]).tolist()}')

# Create DataLoaders
BS = CONFIG['stage_a']['phase1']['batch_size']
NW = CONFIG['stage_a']['phase1']['num_workers']

train_loader = DataLoader(train_dataset, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=True)

print(f'\nDataLoaders created (batch_size={BS}, num_workers={NW})')
print(f'  Train batches/epoch: {len(train_loader)}')
print(f'  Val batches:         {len(val_loader)}')

# Estimate training time
print(f'\n--- Training Time Estimate ---')
est_seconds_per_batch = 0.5  # rough estimate for P100
batches_per_epoch = len(train_loader)
total_epochs = CONFIG['stage_a']['phase1']['epochs'] + CONFIG['stage_a']['phase2']['epochs']
total_batches = batches_per_epoch * total_epochs
est_hours = total_batches * est_seconds_per_batch / 3600
print(f'  Batches/epoch: {batches_per_epoch}')
print(f'  Total epochs (P1+P2): {total_epochs}')
print(f'  Est. ~{est_seconds_per_batch}s/batch -> ~{est_hours:.1f} hours total')
if est_hours > 11:
    print(f'  WARNING: May exceed 12h Kaggle session! Consider reducing epochs or increasing subsample.')

## 3. Load Pre-trained Checkpoint

In [ ]:
# ============================================================================
# CELL 8: Load Pre-trained Checkpoint + Resume Detection
# ============================================================================

model = SiamUnet()

# ── Resume Detection ──────────────────────────────────────────────────────
# If a phase checkpoint exists in OUTPUT_DIR, resume from it.
# Otherwise load the original pre-trained checkpoint.

def find_latest_phase_checkpoint(directories, phase):
    candidates = []
    for d in directories:
        if not d.exists(): continue
        candidates.extend(list(d.rglob(f'phase{phase}_epoch*.pth.tar')))
    if not candidates:
        return None
    candidates = sorted(
        candidates,
        key=lambda p: int(p.name.split('_epoch')[1].split('.')[0])
    )
    return candidates[-1]

# Cari di /kaggle/working/output ATAU /kaggle/input (jika di-upload sbg dataset)
search_dirs = [OUTPUT_DIR, Path('/kaggle/input')]
resume_phase1_from = find_latest_phase_checkpoint(search_dirs, phase=1)
resume_phase2_from = find_latest_phase_checkpoint(search_dirs, phase=2)

if resume_phase2_from:
    load_path = resume_phase2_from
    resume_mode = 'phase2'
elif resume_phase1_from:
    load_path = resume_phase1_from
    resume_mode = 'phase1'
else:
    load_path = CHECKPOINT_PATH
    resume_mode = 'pretrained'

print(f'Resume mode: {resume_mode}')
print(f'Loading from: {load_path}')

try:
    ckpt = torch.load(load_path, map_location='cpu', weights_only=True)
except Exception:
    ckpt = torch.load(load_path, map_location='cpu', weights_only=False)

if isinstance(ckpt, dict):
    state_dict = ckpt.get('state_dict') or ckpt.get('model_state_dict') or ckpt.get('model') or ckpt
else:
    state_dict = ckpt

cleaned = {}
for k, v in state_dict.items():
    k_clean = k
    for prefix in ('module.', 'model.'):
        if k_clean.startswith(prefix):
            k_clean = k_clean[len(prefix):]
    cleaned[k_clean] = v

result = model.load_state_dict(cleaned, strict=True)
epoch_loaded = ckpt.get('epoch', 'N/A') if isinstance(ckpt, dict) else 'N/A'
best_f1_loaded = ckpt.get('best_f1', 'N/A') if isinstance(ckpt, dict) else 'N/A'
print(f'  Epoch:          {epoch_loaded}')
print(f'  Best F1:        {best_f1_loaded}')
print(f'  Missing keys:   {result.missing_keys}')
print(f'  State dict keys: {len(cleaned)}')

# Store resume state for Phase 1 / Phase 2 cells
RESUME_PHASE1_EPOCH = 0
RESUME_PHASE2_EPOCH = 0
RESUME_BEST_SEG_ACC = 0.0
RESUME_BEST_MACRO_F1 = 0.0

if resume_mode == 'phase1' and isinstance(ckpt, dict):
    RESUME_PHASE1_EPOCH = ckpt.get('epoch', 0)
    RESUME_BEST_SEG_ACC = ckpt.get('best_f1', 0.0)
    print(f'  -> Resuming Phase 1 from epoch {RESUME_PHASE1_EPOCH}')
elif resume_mode == 'phase2' and isinstance(ckpt, dict):
    RESUME_PHASE1_EPOCH = CONFIG['stage_a']['phase1']['epochs']  # phase1 done
    RESUME_PHASE2_EPOCH = ckpt.get('epoch', 0)
    RESUME_BEST_MACRO_F1 = ckpt.get('best_f1', 0.0)
    print(f'  -> Phase 1 done. Resuming Phase 2 from epoch {RESUME_PHASE2_EPOCH}')

model = model.to(DEVICE)
print(f'Model moved to {DEVICE}')


## 4. Training Functions

In [ ]:
# ============================================================================
# CELL 9: Training & Evaluation Functions
# ============================================================================

def freeze_layers(model, prefixes):
    """Freeze parameters whose names start with any of the given prefixes."""
    frozen = 0
    for name, param in model.named_parameters():
        if any(name.startswith(p) for p in prefixes):
            param.requires_grad = False
            frozen += 1
        else:
            param.requires_grad = True
    trainable = sum(1 for p in model.parameters() if p.requires_grad)
    print(f'  Frozen: {frozen} params, Trainable: {trainable} params')


def save_checkpoint(model, optimizer, epoch, metrics, path, is_best=False):
    """Save checkpoint in backend-compatible format.
    
    Format MUST match what ml_inference.py expects:
      - dict with key 'state_dict'
      - NO 'module.' prefix on keys
    """
    state = {
        'epoch': epoch,
        'state_dict': model.state_dict(),  # raw keys, no prefix
        'optimizer': optimizer.state_dict(),
        'val_f1_avg': metrics.get('macro_f1', 0.0),
        'best_f1': metrics.get('best_f1', 0.0),
        'config': CONFIG,
    }
    torch.save(state, path)
    if is_best:
        best_path = path.parent / 'model_best.pth.tar'
        torch.save(state, best_path)
        print(f'  Saved best checkpoint: {best_path}')


def train_one_epoch(model, loader, optimizer, seg_criterion, dmg_criterion, weights_loss, device, step_counter=0):
    """Train for one epoch.
    
    Args:
        weights_loss: [w_seg_pre, w_seg_post, w_damage]
    """
    model.train()
    running_loss = 0.0
    n_batches = 0
    save_interval = CONFIG['compute']['checkpoint_save_interval']
    log_interval = CONFIG['compute']['log_interval']
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for batch in pbar:
        pre = batch['pre'].to(device)
        post = batch['post'].to(device)
        seg_pre_gt = batch['seg_pre'].to(device)
        seg_post_gt = batch['seg_post'].to(device)
        damage_gt = batch['damage'].to(device)
        
        optimizer.zero_grad()
        seg_pre_pred, seg_post_pred, damage_pred = model(pre, post)
        
        # Compute weighted loss
        loss = 0.0
        if weights_loss[0] > 0:
            loss += weights_loss[0] * seg_criterion(seg_pre_pred, seg_pre_gt)
        if weights_loss[1] > 0:
            loss += weights_loss[1] * seg_criterion(seg_post_pred, seg_post_gt)
        if weights_loss[2] > 0:
            loss += weights_loss[2] * dmg_criterion(damage_pred, damage_gt)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        n_batches += 1
        step_counter += 1
        
        if step_counter % log_interval == 0:
            pbar.set_postfix({'loss': f'{running_loss/n_batches:.4f}', 'step': step_counter})
        
        # Periodic checkpoint save (Kaggle session safety)
        if step_counter % save_interval == 0:
            ckpt_path = OUTPUT_DIR / f'checkpoint_step{step_counter}.pth.tar'
            save_checkpoint(model, optimizer, -1, {'macro_f1': 0.0, 'best_f1': 0.0}, ckpt_path)
    
    return running_loss / max(n_batches, 1), step_counter


@torch.no_grad()
def evaluate(model, loader, seg_criterion, dmg_criterion, weights_loss, device, mode='val'):
    """Evaluate model on val/test set.
    
    Returns:
        dict with loss, accuracy, macro_f1, per_class metrics
    """
    model.eval()
    running_loss = 0.0
    n_batches = 0
    
    # For metrics computation
    all_damage_pred = []
    all_damage_gt = []
    all_seg_correct = 0
    all_seg_total = 0
    
    pbar = tqdm(loader, desc=f'Evaluating ({mode})', leave=False)
    for batch in pbar:
        pre = batch['pre'].to(device)
        post = batch['post'].to(device)
        seg_pre_gt = batch['seg_pre'].to(device)
        seg_post_gt = batch['seg_post'].to(device)
        damage_gt = batch['damage'].to(device)
        
        seg_pre_pred, seg_post_pred, damage_pred = model(pre, post)
        
        # Loss
        loss = 0.0
        if weights_loss[0] > 0:
            loss += weights_loss[0] * seg_criterion(seg_pre_pred, seg_pre_gt)
        if weights_loss[1] > 0:
            loss += weights_loss[1] * seg_criterion(seg_post_pred, seg_post_gt)
        if weights_loss[2] > 0:
            loss += weights_loss[2] * dmg_criterion(damage_pred, damage_gt)
        
        running_loss += loss.item()
        n_batches += 1
        
        # Segmentation accuracy
        seg_pred = seg_post_pred.argmax(dim=1)
        all_seg_correct += (seg_pred == seg_post_gt).sum().item()
        all_seg_total += seg_post_gt.numel()
        
        # Damage predictions (only non-background pixels)
        dmg_pred = damage_pred.argmax(dim=1)
        mask = damage_gt > 0  # exclude background pixels
        if mask.any():
            all_damage_pred.extend(dmg_pred[mask].cpu().numpy().tolist())
            all_damage_gt.extend(damage_gt[mask].cpu().numpy().tolist())
    
    avg_loss = running_loss / max(n_batches, 1)
    seg_accuracy = all_seg_correct / max(all_seg_total, 1)
    
    # Compute per-class F1 for damage
    damage_metrics = compute_damage_metrics(all_damage_gt, all_damage_pred)
    
    return {
        'loss': avg_loss,
        'seg_accuracy': seg_accuracy,
        **damage_metrics,
    }


def compute_damage_metrics(y_true, y_pred):
    """Compute per-class and macro F1 for damage classification."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    if len(y_true) == 0:
        return {'macro_f1': 0.0, 'accuracy': 0.0, 'per_class_f1': {}}
    
    classes = [1, 2, 3, 4]  # exclude background
    class_names = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
    
    f1_values = []
    per_class = {}
    
    for cls, name in zip(classes, class_names):
        tp = int(((y_pred == cls) & (y_true == cls)).sum())
        fp = int(((y_pred == cls) & (y_true != cls)).sum())
        fn = int(((y_pred != cls) & (y_true == cls)).sum())
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        per_class[name] = {'precision': precision, 'recall': recall, 'f1': f1}
        if (y_true == cls).sum() > 0:  # only include classes that exist in ground truth
            f1_values.append(f1)
    
    accuracy = float((y_pred == y_true).sum() / len(y_true))
    macro_f1 = float(np.mean(f1_values)) if f1_values else 0.0
    
    return {
        'macro_f1': macro_f1,
        'accuracy': accuracy,
        'per_class_f1': per_class,
        'n_samples': len(y_true),
    }


print('Training functions defined.')

## 5. Phase 1: Building Segmentation

In [ ]:
# ============================================================================
# CELL 10: Phase 1 — Building Segmentation Training (with Resume)
# ============================================================================
p1_total_epochs = CONFIG['stage_a']['phase1']['epochs']

if RESUME_PHASE1_EPOCH >= p1_total_epochs:
    print(f'Phase 1 already complete ({RESUME_PHASE1_EPOCH}/{p1_total_epochs} epochs). Skipping.')
    history_p1 = {'train_loss': [], 'val_loss': [], 'val_seg_acc': [], 'val_iou': []}
    best_seg_acc = RESUME_BEST_SEG_ACC
    # Load best phase1 model weights for phase 2
    best_p1_path = OUTPUT_DIR / 'model_best.pth.tar'
    if best_p1_path.exists():
        ck = torch.load(best_p1_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['state_dict'])
        model = model.to(DEVICE)
        print(f'Loaded best Phase 1 checkpoint for Phase 2 use.')
else:
    print('='*60)
    print('PHASE 1: Building Segmentation')
    print(f'  Resume from epoch: {RESUME_PHASE1_EPOCH}')
    print(f'  Remaining: {p1_total_epochs - RESUME_PHASE1_EPOCH} epochs')
    print('='*60)

    freeze_layers(model, DAMAGE_LAYER_PREFIXES)

    seg_weights = torch.tensor(CONFIG['stage_a']['phase1']['class_weights_seg'], dtype=torch.float32).to(DEVICE)
    seg_criterion = nn.CrossEntropyLoss(weight=seg_weights)
    dmg_criterion = nn.CrossEntropyLoss()

    p1_config = CONFIG['stage_a']['phase1']
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=p1_config['lr'],
        weight_decay=p1_config['weight_decay'],
    )

    if RESUME_PHASE1_EPOCH > 0 and resume_mode == 'phase1' and isinstance(ckpt, dict) and 'optimizer' in ckpt:
        try:
            optimizer.load_state_dict(ckpt['optimizer'])
            print('  Optimizer state restored.')
        except Exception as e:
            print(f'  Could not restore optimizer: {e}')

    weights_loss = p1_config['weights_loss']
    history_p1 = {'train_loss': [], 'val_loss': [], 'val_seg_acc': [], 'val_iou': []}
    best_seg_acc = RESUME_BEST_SEG_ACC
    step_counter = 0

    # Auto-backup: copy checkpoint to /kaggle/working/ root for easy download
    def backup_checkpoint(ckpt_path):
        import shutil
        dst = Path('/kaggle/working') / ckpt_path.name
        try:
            shutil.copy2(ckpt_path, dst)
        except Exception:
            pass  # ignore if not on Kaggle

    print(f'Training Phase 1: epoch {RESUME_PHASE1_EPOCH+1} to {p1_total_epochs}')

    for epoch in range(RESUME_PHASE1_EPOCH + 1, p1_total_epochs + 1):
        t_start = time.time()
        train_loss, step_counter = train_one_epoch(
            model, train_loader, optimizer, seg_criterion, dmg_criterion,
            weights_loss, DEVICE, step_counter
        )
        val_metrics = evaluate(
            model, val_loader, seg_criterion, dmg_criterion,
            weights_loss, DEVICE, mode='val'
        )
        history_p1['train_loss'].append(train_loss)
        history_p1['val_loss'].append(val_metrics['loss'])
        history_p1['val_seg_acc'].append(val_metrics['seg_accuracy'])
        elapsed = time.time() - t_start
        is_best = val_metrics['seg_accuracy'] > best_seg_acc
        if is_best:
            best_seg_acc = val_metrics['seg_accuracy']
        print(f'Epoch {epoch:2d}/{p1_total_epochs} | '
              f'Train: {train_loss:.4f} | '
              f'Val Loss: {val_metrics["loss"]:.4f} | '
              f'Seg Acc: {val_metrics["seg_accuracy"]:.4f} | '
              f'{elapsed:.0f}s'
              f'{" *BEST*" if is_best else ""}')
        ckpt_path = OUTPUT_DIR / f'phase1_epoch{epoch}.pth.tar'
        save_checkpoint(
            model, optimizer, epoch,
            {'macro_f1': val_metrics.get('macro_f1', 0.0), 'best_f1': best_seg_acc},
            ckpt_path,
            is_best=is_best,
        )
        backup_checkpoint(ckpt_path)  # copy to /kaggle/working/ for easy download
    print(f'Phase 1 complete. Best seg accuracy: {best_seg_acc:.4f}')


## 6. Phase 2: Damage Classification

In [ ]:
# ============================================================================
# CELL 11: Phase 2 — Damage Classification Training (with Resume)
# ============================================================================
p2_total_epochs = CONFIG['stage_a']['phase2']['epochs']

print('='*60)
print('PHASE 2: Damage Classification')
if RESUME_PHASE2_EPOCH > 0:
    print(f'  Resume from epoch: {RESUME_PHASE2_EPOCH}')
    print(f'  Remaining: {p2_total_epochs - RESUME_PHASE2_EPOCH} epochs')
else:
    best_p1_path = OUTPUT_DIR / 'model_best.pth.tar'
    if best_p1_path.exists():
        p1_ckpt = torch.load(best_p1_path, map_location='cpu', weights_only=False)
        model.load_state_dict(p1_ckpt['state_dict'])
        print(f'  Loaded best Phase 1 checkpoint (epoch {p1_ckpt.get("epoch","?")})')
print('='*60)

model = model.to(DEVICE)

SEG_LAYER_PREFIXES = [
    'encoder1', 'encoder2', 'encoder3', 'encoder4',
    'bottleneck', 'pool',
    'upconv4.', 'decoder4', 'upconv3.', 'decoder3',
    'upconv2.', 'decoder2', 'upconv1.', 'decoder1',
    'conv_s',
]
freeze_layers(model, SEG_LAYER_PREFIXES)

p2_config = CONFIG['stage_a']['phase2']
dmg_weights = torch.tensor(p2_config['class_weights_dmg'], dtype=torch.float32).to(DEVICE)
dmg_criterion = nn.CrossEntropyLoss(weight=dmg_weights)
seg_criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=p2_config['lr'],
    weight_decay=p2_config['weight_decay'],
)

if RESUME_PHASE2_EPOCH > 0 and resume_mode == 'phase2' and isinstance(ckpt, dict) and 'optimizer' in ckpt:
    try:
        optimizer.load_state_dict(ckpt['optimizer'])
        print('  Optimizer state restored.')
    except Exception as e:
        print(f'  Could not restore optimizer: {e}')

weights_loss = p2_config['weights_loss']
history_p2 = {
    'train_loss': [], 'val_loss': [],
    'val_macro_f1': [], 'val_accuracy': [],
    'val_per_class_f1': [],
}
best_macro_f1 = RESUME_BEST_MACRO_F1
step_counter = 0

print(f'Training Phase 2: epoch {RESUME_PHASE2_EPOCH+1} to {p2_total_epochs}')

for epoch in range(RESUME_PHASE2_EPOCH + 1, p2_total_epochs + 1):
    t_start = time.time()
    train_loss, step_counter = train_one_epoch(
        model, train_loader, optimizer, seg_criterion, dmg_criterion,
        weights_loss, DEVICE, step_counter
    )
    val_metrics = evaluate(
        model, val_loader, seg_criterion, dmg_criterion,
        weights_loss, DEVICE, mode='val'
    )
    history_p2['train_loss'].append(train_loss)
    history_p2['val_loss'].append(val_metrics['loss'])
    history_p2['val_macro_f1'].append(val_metrics['macro_f1'])
    history_p2['val_accuracy'].append(val_metrics.get('accuracy', 0))
    history_p2['val_per_class_f1'].append(val_metrics.get('per_class_f1', {}))
    elapsed = time.time() - t_start
    is_best = val_metrics['macro_f1'] > best_macro_f1
    if is_best:
        best_macro_f1 = val_metrics['macro_f1']
    pcf1 = val_metrics.get('per_class_f1', {})
    f1_str = ' | '.join([f"{k[:3]}={v['f1']:.3f}" for k, v in pcf1.items()])
    print(f'Epoch {epoch:2d}/{p2_total_epochs} | '
          f'Train: {train_loss:.4f} | '
          f'Val: {val_metrics["loss"]:.4f} | '
          f'F1: {val_metrics["macro_f1"]:.4f} | '
          f'{f1_str} | '
          f'{elapsed:.0f}s'
          f'{" *BEST*" if is_best else ""}')
    save_checkpoint(
        model, optimizer, epoch,
        {'macro_f1': val_metrics['macro_f1'], 'best_f1': best_macro_f1},
        OUTPUT_DIR / f'phase2_epoch{epoch}.pth.tar',
        is_best=is_best,
    )

print(f'Phase 2 complete. Best macro F1: {best_macro_f1:.4f}')


## 7. Evaluation on Test Set

In [ ]:
# ============================================================================
# CELL 12: Final Evaluation on Test Set
# ============================================================================
print('='*60)
print('FINAL EVALUATION ON TEST SET')
print('='*60)

# Load best checkpoint
best_path = OUTPUT_DIR / 'model_best.pth.tar'
best_ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
model.load_state_dict(best_ckpt['state_dict'])
model = model.to(DEVICE)
print(f'Loaded best checkpoint (best_f1={best_ckpt.get("best_f1", "?"):.4f})')

# Evaluate
test_metrics = evaluate(
    model, test_loader, seg_criterion, dmg_criterion,
    [0.0, 0.0, 1.0], DEVICE, mode='test'
)

print(f'\n--- Test Set Results ---')
print(f'  Loss:        {test_metrics["loss"]:.4f}')
print(f'  Seg Accuracy:{test_metrics["seg_accuracy"]:.4f}')
print(f'  Macro F1:    {test_metrics["macro_f1"]:.4f}')
print(f'  Accuracy:    {test_metrics.get("accuracy", 0):.4f}')
print(f'  N samples:   {test_metrics.get("n_samples", 0):,}')
print(f'\nPer-class F1:')
for cls_name, metrics in test_metrics.get('per_class_f1', {}).items():
    print(f'  {cls_name:15s}: P={metrics["precision"]:.3f} R={metrics["recall"]:.3f} F1={metrics["f1"]:.3f}')

# Compare with baseline
print(f'\n--- Comparison ---')
print(f'  Baseline (zero-shot on Beirut): Macro F1 = 0.221')
print(f'  After fine-tuning (xBD test):   Macro F1 = {test_metrics["macro_f1"]:.4f}')
improvement = (test_metrics['macro_f1'] - 0.221) / 0.221 * 100
print(f'  Improvement: {improvement:+.1f}%')

## 8. Visualization

In [ ]:
# ============================================================================
# CELL 13: Visualization — Loss Curves, Confusion Matrix, ROC
# ============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

DAMAGE_CMAP = ListedColormap([
    [0.2, 0.2, 0.2, 1.0],
    [0.18, 0.80, 0.44, 1.0],
    [0.95, 0.77, 0.06, 1.0],
    [0.90, 0.49, 0.13, 1.0],
    [0.91, 0.30, 0.24, 1.0],
])

# --- Loss Curves ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Phase 1
ax = axes[0]
ax.plot(history_p1['train_loss'], 'b-', label='Train', linewidth=2)
ax.plot(history_p1['val_loss'], 'r-', label='Val', linewidth=2)
ax.set_title('Phase 1: Segmentation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Phase 2
ax = axes[1]
ax.plot(history_p2['train_loss'], 'b-', label='Train', linewidth=2)
ax.plot(history_p2['val_loss'], 'r-', label='Val', linewidth=2)
ax2 = ax.twinx()
ax2.plot(history_p2['val_macro_f1'], 'g--', label='Macro F1', linewidth=2)
ax.set_title('Phase 2: Damage Classification Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax2.set_ylabel('Macro F1', color='green')
ax.legend(loc='upper right')
ax2.legend(loc='center right')
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Loss curves saved.')

# --- Confusion Matrix (from test evaluation) ---
# Re-run evaluation collecting all predictions for confusion matrix
model.eval()
all_pred = []
all_true = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Collecting predictions'):
        pre = batch['pre'].to(DEVICE)
        post = batch['post'].to(DEVICE)
        damage_gt = batch['damage']
        
        _, _, damage_pred = model(pre, post)
        dmg_pred = damage_pred.argmax(dim=1).cpu()
        
        mask = damage_gt > 0
        if mask.any():
            all_pred.extend(dmg_pred[mask].numpy().tolist())
            all_true.extend(damage_gt[mask].numpy().tolist())

# Build confusion matrix
cm = np.zeros((5, 5), dtype=np.int64)
for t, p in zip(all_true, all_pred):
    if 0 <= t < 5 and 0 <= p < 5:
        cm[t, p] += 1

# Normalize by row
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=DAMAGE_CLASSES, yticklabels=DAMAGE_CLASSES,
    square=True, linewidths=0.5, ax=ax,
    vmin=0, vmax=1,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Normalized) — Test Set')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')

In [ ]:
# ============================================================================
# CELL 14: Sample Prediction Visualizations
# ============================================================================

model.eval()
n_samples = min(5, len(test_dataset))

fig, axes = plt.subplots(n_samples, 4, figsize=(20, 5 * n_samples))
if n_samples == 1:
    axes = axes.reshape(1, -1)

sample_indices = np.linspace(0, len(test_dataset) - 1, n_samples, dtype=int)

for i, idx in enumerate(sample_indices):
    sample = test_dataset[idx]
    
    pre = sample['pre'].unsqueeze(0).to(DEVICE)
    post = sample['post'].unsqueeze(0).to(DEVICE)
    gt_damage = sample['damage'].numpy()
    
    with torch.no_grad():
        _, _, damage_pred = model(pre, post)
        pred_mask = damage_pred.argmax(dim=1).squeeze().cpu().numpy()
    
    # Display
    pre_img = sample['pre'].numpy().transpose(1, 2, 0)
    post_img = sample['post'].numpy().transpose(1, 2, 0)
    
    axes[i, 0].imshow(np.clip(pre_img, 0, 1))
    axes[i, 0].set_title('Pre-Disaster')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(np.clip(post_img, 0, 1))
    axes[i, 1].set_title('Post-Disaster')
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(gt_damage, cmap=DAMAGE_CMAP, vmin=0, vmax=4, interpolation='nearest')
    axes[i, 2].set_title('Ground Truth')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(pred_mask, cmap=DAMAGE_CMAP, vmin=0, vmax=4, interpolation='nearest')
    axes[i, 3].set_title('Prediction')
    axes[i, 3].axis('off')

fig.suptitle('Sample Predictions — Test Set', fontsize=16, fontweight='bold')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample predictions saved.')

## 9. Checkpoint Compatibility Verification

In [ ]:
# ============================================================================
# CELL 15: Final Checkpoint Compatibility Test
# ============================================================================
print('='*60)
print('CHECKPOINT COMPATIBILITY TEST')
print('Backend: ml_inference.py')
print('='*60)

best_path = OUTPUT_DIR / 'model_best.pth.tar'

# Test 1: Load checkpoint
try:
    ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
    print('[PASS] Checkpoint loads with map_location="cpu"')
except Exception as e:
    print(f'[FAIL] Cannot load checkpoint: {e}')

# Test 2: Has 'state_dict' key
assert 'state_dict' in ckpt, 'Missing state_dict key!'
print(f'[PASS] Has "state_dict" key ({len(ckpt["state_dict"])} params)')

# Test 3: No prefix
has_prefix = any(k.startswith(('module.', 'model.')) for k in ckpt['state_dict'])
assert not has_prefix, 'Keys have module./model. prefix!'
print('[PASS] No module./model. prefix on keys')

# Test 4: Strict load
test_model = SiamUnet()
result = test_model.load_state_dict(ckpt['state_dict'], strict=True)
assert len(result.missing_keys) == 0, f'Missing keys: {result.missing_keys}'
print(f'[PASS] strict=True load: 0 missing, 0 unexpected')

# Test 5: Forward pass
test_model.eval()
dummy_pre = torch.randn(1, 3, 256, 256)
dummy_post = torch.randn(1, 3, 256, 256)
with torch.no_grad():
    seg_pre, seg_post, damage = test_model(dummy_pre, dummy_post)
assert seg_pre.shape == (1, 2, 256, 256)
assert seg_post.shape == (1, 2, 256, 256)
assert damage.shape == (1, 5, 256, 256)
print(f'[PASS] Forward pass: seg={seg_pre.shape}, damage={damage.shape}')

# Test 6: Softmax averaging (matches backend)
probs = torch.softmax(damage, dim=1).mean(dim=(0, 2, 3)).detach()
assert probs.shape == (5,)
print(f'[PASS] Softmax averaging: shape={probs.shape}')

print('\n' + '='*60)
print('ALL COMPATIBILITY TESTS PASSED')
print('Checkpoint is ready for backend deployment.')
print('='*60)

# Print deployment instructions
print(f'\n--- Deployment Instructions ---')
print(f'1. Download: {best_path}')
print(f'2. Copy to: data/raw/models/microsoft_building_damage_assessment_cnn_siamese/')
print(f'3. Update .env: ML_MODEL_CHECKPOINT_PATH=../data/raw/models/.../model_best.pth.tar')
print(f'4. Restart backend: uvicorn app.main:app --reload')

In [ ]:
# ============================================================================
# CELL 16: Save Training Summary
# ============================================================================

summary = {
    'platform': 'Kaggle' if IS_KAGGLE else 'Local',
    'device': str(DEVICE),
    'pytorch_version': torch.__version__,
    'seed': SEED,
    'data': {
        'train_pairs': len(train_pairs),
        'tier3_pairs': len(tier3_pairs),
        'total_pairs': len(all_pairs),
        'train_patches': len(train_dataset),
        'val_patches': len(val_dataset),
        'test_patches': len(test_dataset),
        'subsample_tier3': CONFIG['compute']['subsample_tier3'],
    },
    'phase1': {
        'epochs': len(history_p1['train_loss']),
        'best_seg_accuracy': best_seg_acc,
        'final_train_loss': history_p1['train_loss'][-1] if history_p1['train_loss'] else None,
        'final_val_loss': history_p1['val_loss'][-1] if history_p1['val_loss'] else None,
    },
    'phase2': {
        'epochs': len(history_p2['train_loss']),
        'best_macro_f1': best_macro_f1,
        'final_train_loss': history_p2['train_loss'][-1] if history_p2['train_loss'] else None,
        'final_val_loss': history_p2['val_loss'][-1] if history_p2['val_loss'] else None,
    },
    'test_results': test_metrics,
    'baseline_macro_f1': 0.221,
    'config': CONFIG,
}

with open(OUTPUT_DIR / 'training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('Training summary saved to:', OUTPUT_DIR / 'training_summary.json')
print('\nDone! Download model_best.pth.tar for backend deployment.')